# Results — TTN, SigLIP2, Popularity

Recall@10 / Recall@100 for the three built models, on a shared held-out
test sample, scored the same way throughout: candidates restricted to
each pair's target category, hit@k by top-k **membership**
(`target in topk(scores)`) — never `(scores > true_score).sum()`, which
credits every zero-vector tie as a free hit.

**Prerequisites** — each build script writes into `data/tower/`, read
here:
- `TTN/build_data.py` then `TTN/build_model.py` (optionally
  `TTN/encode_descriptions.py` in between) → `ttn_complementary.pt`,
  `items.npz`, `pairs_{train,test}.parquet`, `node_of_item.npy`
- `SigLIP2/build_siglip2.py` → `siglip_img_emb.npy`,
  `siglip_img_status.npy`
- `Popularity/build_popularity.py` → `popularity_top100.json`

## Load the shared test data

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().resolve()
if ROOT.name != "RecSystem":
    ROOT = ROOT.parent
OUT_DIR = ROOT / "data" / "tower"
CAT_ORDER = ["cat_2", "cat_3", "cat_4", "brand", "color", "material", "product_type", "features"]
N_SAMPLE = 20_000
SEED = 0
BUCKET_ORDER = ["never", "1-5", "6-25", "26-100", ">100", "all"]

node_of_item = np.load(OUT_DIR / "node_of_item.npy")
pairs_train = pd.read_parquet(OUT_DIR / "pairs_train.parquet")
pairs_test = pd.read_parquet(OUT_DIR / "pairs_test.parquet")
n_items = len(node_of_item)
print(f"items: {n_items:,} | train pairs: {len(pairs_train):,} | test pairs: {len(pairs_test):,}")

In [ ]:
# --- in-node candidate pools, and target-frequency buckets -----------------
order = np.argsort(node_of_item, kind="stable")
starts = np.searchsorted(node_of_item[order], np.arange(int(node_of_item.max()) + 2))

def cands_for_node(node_id):
    return order[starts[node_id]:starts[node_id + 1]]

target_freq_train = pairs_train.groupby("target_idx").size()

def bucket(f):
    if f == 0: return "never"
    if f <= 5: return "1-5"
    if f <= 25: return "6-25"
    if f <= 100: return "26-100"
    return ">100"

rng = np.random.default_rng(SEED)
take = rng.choice(len(pairs_test), min(N_SAMPLE, len(pairs_test)), replace=False)
sample = pairs_test.iloc[take].reset_index(drop=True)
sample["freq"] = sample["target_idx"].map(target_freq_train).fillna(0).astype(int)
sample["bucket"] = sample["freq"].apply(bucket)
print(f"test sample: {len(sample):,} pairs (seed {SEED})")
print(sample["bucket"].value_counts().reindex(BUCKET_ORDER[:-1]))

## Score Popularity — no query read, fixed per-category top-10/top-100

In [ ]:
pop = json.load(open(OUT_DIR / "popularity_top100.json"))
top10_by_node = {int(k): set(v) for k, v in pop["top10_by_node"].items()}
top100_by_node = {int(k): set(v) for k, v in pop["top100_by_node"].items()}

hit10 = {}
hit100 = {}
hit10["POP"] = np.zeros(len(sample), dtype=bool)
hit100["POP"] = np.zeros(len(sample), dtype=bool)
for node_id, grp in sample.groupby("target_node_id"):
    p10, p100 = top10_by_node.get(int(node_id), set()), top100_by_node.get(int(node_id), set())
    rows = grp.index.to_numpy()
    for r, t in enumerate(grp["target_idx"].to_numpy()):
        hit10["POP"][rows[r]] = int(t) in p10
        hit100["POP"][rows[r]] = int(t) in p100
print("Popularity scored")

## Score SigLIP2 — pure image cosine similarity, no training

This is the carousel's actual retrieval logic: cosine similarity over
frozen SigLIP2 image embeddings alone. An item with no usable image
(about 28% of the catalogue) can't be ranked by this signal and scores a
miss, not a free hit.

In [ ]:
img_emb = np.load(OUT_DIR / "siglip_img_emb.npy").astype("float32")
img_status = np.load(OUT_DIR / "siglip_img_status.npy")
has_img = img_status == 1
img_norm = img_emb / (np.linalg.norm(img_emb, axis=1, keepdims=True) + 1e-9)

hit10["SigLIP2"] = np.zeros(len(sample), dtype=bool)
hit100["SigLIP2"] = np.zeros(len(sample), dtype=bool)
for node_id, grp in sample.groupby("target_node_id"):
    node_id = int(node_id)
    cand = cands_for_node(node_id)
    if len(cand) < 2:
        continue
    rows = grp.index.to_numpy()
    q_idx = grp["query_idx"].to_numpy()
    t_idx = grp["target_idx"].to_numpy()
    cand_pos = {c: i for i, c in enumerate(cand)}
    t_pos = np.array([cand_pos.get(t, -1) for t in t_idx])

    scores = img_norm[q_idx] @ img_norm[cand].T
    mask = np.outer(has_img[q_idx], has_img[cand])
    scores = np.where(mask, scores, -np.inf)
    self_mask = cand[None, :] == q_idx[:, None]
    scores = np.where(self_mask, -np.inf, scores)

    k = min(100, scores.shape[1])
    top_idx = np.argpartition(-scores, kth=min(9, k - 1), axis=1)[:, :k]
    order10 = np.argsort(-np.take_along_axis(scores, top_idx, axis=1), axis=1)
    ranked = np.take_along_axis(top_idx, order10, axis=1)
    for r, tp in enumerate(t_pos):
        if tp < 0:
            continue
        in_top = ranked[r]
        if len(in_top) == 0 or scores[r, in_top[0]] == -np.inf:
            continue
        hit10["SigLIP2"][rows[r]] = tp in set(in_top[:10])
        hit100["SigLIP2"][rows[r]] = tp in set(in_top[:100])
print("SigLIP2 scored")

## Score TTN — the trained two-tower model

In [ ]:
CHECKPOINT = OUT_DIR / "ttn_complementary.pt"
ck = torch.load(CHECKPOINT, weights_only=False)
cfg = ck["config"]
DEVICE = "mps" if torch.backends.mps.is_available() else (
         "cuda" if torch.cuda.is_available() else "cpu")

arrays = dict(np.load(OUT_DIR / "items.npz"))
title_t = torch.tensor(arrays["title_emb"], device=DEVICE)
USE_DESCRIPTION = cfg["use_description"]
desc_t = (torch.tensor(arrays["desc_emb"], device=DEVICE) if USE_DESCRIPTION
          else torch.zeros(n_items, 1, device=DEVICE))
USE_IMAGE = cfg.get("use_image", False)
img_t = torch.tensor(img_emb, device=DEVICE) if USE_IMAGE else torch.zeros(n_items, 1, device=DEVICE)
cat_t = torch.tensor(arrays["cat_ids"], device=DEVICE)
num_t = torch.tensor(arrays["numeric"], device=DEVICE)
mu, sd = ck["numeric_standardisation"]["mean"].to(DEVICE), ck["numeric_standardisation"]["std"].to(DEVICE)
num_t = (num_t - mu) / sd
CAT_DIM, NODE_DIM, HIDDEN, OUT_DIM = cfg["cat_dim"], cfg["node_dim"], cfg["hidden"], cfg["out_dim"]
vocab_sizes = ck["vocab_sizes"]


class ProductEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[name] + 1, CAT_DIM, padding_idx=0)
            for name in CAT_ORDER])
        self.title = nn.Linear(title_t.shape[1], 128)
        self.description = nn.Linear(desc_t.shape[1], 128) if USE_DESCRIPTION else None
        self.image = nn.Linear(img_t.shape[1], 128) if USE_IMAGE else None
        self.numeric = nn.Linear(num_t.shape[1], 16)
        self.norm_cat = nn.LayerNorm(len(CAT_ORDER) * CAT_DIM, elementwise_affine=False)
        self.norm_title = nn.LayerNorm(128, elementwise_affine=False)
        self.norm_description = nn.LayerNorm(128, elementwise_affine=False) if USE_DESCRIPTION else None
        self.norm_image = nn.LayerNorm(128, elementwise_affine=False) if USE_IMAGE else None
        self.norm_numeric = nn.LayerNorm(16, elementwise_affine=False)
        self.mlp = nn.Sequential(
            nn.Linear(len(CAT_ORDER) * CAT_DIM + 128
                      + (128 if USE_DESCRIPTION else 0)
                      + (128 if USE_IMAGE else 0) + 16, HIDDEN),
            nn.ReLU(), nn.Linear(HIDDEN, OUT_DIM))

    def forward(self, idx):
        cat = torch.cat([emb(cat_t[idx, j]) for j, emb in enumerate(self.embeddings)], dim=-1)
        parts = [self.norm_cat(cat), self.norm_title(self.title(title_t[idx]))]
        if self.description is not None:
            parts.append(self.norm_description(self.description(desc_t[idx])))
        if self.image is not None:
            parts.append(self.norm_image(self.image(img_t[idx])))
        parts.append(self.norm_numeric(self.numeric(num_t[idx])))
        return self.mlp(torch.cat(parts, dim=-1))


class ComplementaryTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.query_encoder = ProductEncoder()
        self.candidate_encoder = ProductEncoder()
        self.node = nn.Embedding(vocab_sizes["target_node"] + 1, NODE_DIM, padding_idx=0)
        self.query_out = nn.Linear(OUT_DIM + NODE_DIM, OUT_DIM)

    def query(self, idx, node_id):
        h = torch.cat([self.query_encoder(idx), self.node(node_id)], dim=-1)
        return F.normalize(self.query_out(h), dim=-1)

    def candidate(self, idx):
        return F.normalize(self.candidate_encoder(idx), dim=-1)


model = ComplementaryTwoTower().to(DEVICE)
model.load_state_dict(ck["state_dict"])
model.eval()
print(f"TTN checkpoint loaded (best_epoch {cfg.get('best_epoch')}, "
      f"use_description={USE_DESCRIPTION}, use_image={USE_IMAGE})")

In [ ]:
hit10["TTN"] = np.zeros(len(sample), dtype=bool)
hit100["TTN"] = np.zeros(len(sample), dtype=bool)
with torch.no_grad():
    cand_vecs = torch.empty(n_items, OUT_DIM, device=DEVICE)
    for i in range(0, n_items, 8192):
        j = min(i + 8192, n_items)
        cand_vecs[i:j] = model.candidate(torch.arange(i, j, device=DEVICE))
    for node_id, grp in sample.groupby("target_node_id"):
        node_id = int(node_id)
        cand = cands_for_node(node_id)
        if len(cand) < 2:
            continue
        rows = grp.index.to_numpy()
        q_idx = torch.tensor(grp["query_idx"].to_numpy(), device=DEVICE)
        t_idx = grp["target_idx"].to_numpy()
        node_t = torch.full((len(grp),), node_id, device=DEVICE, dtype=torch.long)
        qv = model.query(q_idx, node_t)
        cv = cand_vecs[cand]
        scores = (qv @ cv.T).cpu().numpy()
        self_mask = cand[None, :] == grp["query_idx"].to_numpy()[:, None]
        scores = np.where(self_mask, -np.inf, scores)
        cand_pos = {c: i for i, c in enumerate(cand)}
        for r, t in enumerate(t_idx):
            tp = cand_pos.get(int(t), -1)
            if tp < 0:
                continue
            k = min(100, scores.shape[1])
            top = np.argpartition(-scores[r], kth=min(9, k - 1))[:k]
            top = top[np.argsort(-scores[r, top])]
            hit10["TTN"][rows[r]] = tp in set(top[:10])
            hit100["TTN"][rows[r]] = tp in set(top[:100])
print("TTN scored")

## Results

In [ ]:
COLS = ["TTN", "SigLIP2", "POP"]

def table(hits, title):
    print(f"\n{title}")
    header = f"{'target freq':<12}{'pairs':>8}" + "".join(f"{c:>12}" for c in COLS)
    print(header)
    for b in BUCKET_ORDER:
        m = sample["bucket"] == b if b != "all" else slice(None)
        n = int(m.sum()) if b != "all" else len(sample)
        row = f"{b:<12}{n:>8}"
        for c in COLS:
            v = hits[c][sample.index[m]].mean() if b != "all" else hits[c].mean()
            row += f"{v:>12.4f}"
        print(row)

table(hit10, f"Recall@10  (n={len(sample):,}, seed {SEED})")
table(hit100, f"Recall@100  (n={len(sample):,}, seed {SEED})")